In [5]:
import torch
import deepxde as dde
import numpy as np
import torch.nn as nn 

path = '../neural_model_identification/data_module/trajectories/sample_0'
traj = np.load(path + '/states.npy')
actions = np.load(path + '/actions.npy')
delta_t = 0.3

In [6]:
class NET(nn.Module):
    def __init__(self):
        super(NET, self).__init__()
        self.mlp = torch.nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )
    def forward(self, x):
        return self.mlp(x)

In [7]:
def ode_system(x, y):
    # x is assumed to be my input: [x, y, theta, v, w, t]
    # y is assumed to be my output: [x_next, y_next, theta_next]
    print('x shape: ', x.shape)
    print('y, ', y.shape)
    x_next, y_next, theta_next = y[:, 0], y[:, 1], y[:, 2]

    dy1_t = dde.grad.jacobian(x_next, x, i=0, j=0) # time derivative of X
    dy2_t = dde.grad.jacobian(y_next, x, i=1, j=0) # time derivative of Y
    dy3_t = dde.grad.jacobian(theta_next, x, i=2, j=0) # time derivative of Theta
    
    print('dy1_x  pre selection: ', dy1_t.shape)
    print('-----------------')


    dy1_t = dy1_t.squeeze(-1)
    dy2_t = dy2_t.squeeze(-1)
    dy3_t = dy3_t.squeeze(-1)
    
    print('dy1_x  post selection: ', dy1_t.shape)
    print('-----------------')

    print(f'dy1 - y1: {dy1_t.shape}, {x_next.shape}')
    print(f'dy2 - y2: {dy2_t.shape}, {y_next.shape}')
    print(f'dy3 - y3: {dy3_t.shape}, {theta_next.shape}')

    return [dy1_t - x[:, 3]*np.cos(x[:, 2]), dy2_t - x[:, 3]*np.sin(x[:, 2]), dy3_t - x[:, 4]]

def boundary(x, on_initial):
    return dde.utils.isclose(x[0], 0)

def uniKin(x):
    print('x_shape oriignial : ' , x.shape)
    x, u = x[:, :3], x[:, 3:]
    print('x shape  ', x.shape)

    print('u shape  ', u.shape)

    x_dot = u[:, 0]*np.cos(x[:, 2])
    y_dot = u[:, 0]*np.sin(x[:, 2])
    theta_dot = u[:, 1]
    return np.hstack((x_dot, y_dot, theta_dot))

#---------------------------

cube_dim = np.ones((4, 2))
cube_dim[:, 0] *= -1
geom = dde.geometry.Hypercube(cube_dim[:, 0], cube_dim[:, 1])
timedomain = dde.geometry.TimeDomain(0, delta_t)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)
mlp_ref = NET()

ic1 = dde.icbc.IC(geomtime, lambda x: 0, boundary, component=0)
ic2 = dde.icbc.IC(geomtime, lambda x: 1, boundary, component=1)
ic3 = dde.icbc.IC(geomtime, lambda x: 0, boundary, component=2)
ic4 = dde.icbc.IC(geomtime, lambda x: 1, boundary, component=3)
ic5 = dde.icbc.IC(geomtime, lambda x: 0, boundary, component=4)

ic_list = [ic1]
layer_size = [5] + [50] * 3 + [3]
activation = "tanh"
initializer = "Glorot uniform"
net = dde.nn.FNN(layer_size, activation, initializer)
data = dde.data.PDE(geomtime, ode_system, [],  200, 0, solution=uniKin, num_test=100)
model = dde.Model(data, net)
model.compile("adam", lr=0.001, metrics=["l2 relative error"])
losshistory, train_state = model.train(iterations=20000)

x_shape oriignial :  (200, 5)
x shape   (200, 3)
u shape   (200, 2)
x_shape oriignial :  (162, 5)
x shape   (162, 3)
u shape   (162, 2)
Compiling model...
'compile' took 0.482971 s

Training model...

x shape:  torch.Size([200, 5])
y,  torch.Size([200, 3])


IndexError: tuple index out of range